# Hallucinations and training labels

We inspect animal predictions on empty-labeled test images and animal-labeled training frames where no animal is visible. We compare cameras and sequence positions.


## 1. Setup

Run from `phase1/pair_b` or its `notebooks` directory. Predictions and camera features are recomputed on each run. The README lists the required data and weights.


In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.models import resnet50
from torchvision.ops import FrozenBatchNorm2d
from torchvision.transforms import Compose, Normalize, Resize, ToTensor
from IPython.display import display

APP = Path.cwd()
if APP.name == 'notebooks':
    APP = APP.parent
ROOT = APP.parents[1]
OUT = APP / 'results'
OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(APP / 'src'))
from annotations import load_reviews
from common import load_config, resolve
from common.data import IWildCamImages, load_metadata, normalize
from common.model import load_model

NON_ANIMAL = {'empty', 'motorcycle'}
WEIGHTS = Path.home() / '.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth'


## 2. Metadata


In [2]:
cfg = load_config()
meta = load_metadata(cfg)
data_dir = resolve(cfg['paths']['data_dir'])

train_meta = meta.loc[meta.split.eq('train')]
test_meta = meta.loc[meta.split.isin(['test', 'id_test'])]

train_meta

,split,location_remapped,location,sequence_remapped,seq_id,y,category_id,datetime,filename,image_id,label_name
58259,train,0,0,30191,975e82fe-21bc-11ea-a13a-137349068a90,0,0,2014-06-09 09:32:03.000,87d70644-21bc-11ea-a13a-137349068a90.jpg,87d70644-21bc-11ea-a13a-137349068a90,empty
58260,train,0,0,30191,975e82fe-21bc-11ea-a13a-137349068a90,0,0,2014-06-09 09:32:03.000,975e8182-21bc-11ea-a13a-137349068a90.jpg,975e8182-21bc-11ea-a13a-137349068a90,empty
58261,train,0,0,30191,975e82fe-21bc-11ea-a13a-137349068a90,0,0,2014-06-09 09:32:03.000,934f9414-21bc-11ea-a13a-137349068a90.jpg,934f9414-21bc-11ea-a13a-137349068a90,empty
58262,train,1,1,91,2ffda422-7d42-11eb-8fb5-0242ac1c0002,24,71,2013-01-20 11:34:39.000,8f618aec-21bc-11ea-a13a-137349068a90.jpg,8f618aec-21bc-11ea-a13a-137349068a90,bos taurus
58263,train,1,1,91,2ffda422-7d42-11eb-8fb5-0242ac1c0002,24,71,2013-01-20 11:34:40.000,8d771a62-21bc-11ea-a13a-137349068a90.jpg,8d771a62-21bc-11ea-a13a-137349068a90,bos taurus
...,...,...,...,...,...,...,...,...,...,...,...
188063,train,322,551,33419,9877858c-21bc-11ea-a13a-137349068a90,0,0,2013-03-10 08:00:57.000,978cca38-21bc-11ea-a13a-137349068a90.jpg,978cca38-21bc-11ea-a13a-137349068a90,empty
188064,train,322,551,33419,9877858c-21bc-11ea-a13a-137349068a90,0,0,2013-03-10 08:00:58.000,987783de-21bc-11ea-a13a-137349068a90.jpg,987783de-21bc-11ea-a13a-137349068a90,empty
188065,train,322,551,33930,989e5d4c-21bc-11ea-a13a-137349068a90,52,115,2013-02-20 10:02:15.000,989e5bb2-21bc-11ea-a13a-137349068a90.jpg,989e5bb2-21bc-11ea-a13a-137349068a90,madoqua guentheri
188066,train,322,551,33930,989e5d4c-21bc-11ea-a13a-137349068a90,52,115,2013-02-20 10:02:17.000,9266edd6-21bc-11ea-a13a-137349068a90.jpg,9266edd6-21bc-11ea-a13a-137349068a90,madoqua guentheri


## 3. Test predictions

`id_test` contains new images from training cameras. `test` contains unseen cameras. Here, a model hallucination means predicting an animal when the dataset says empty.


In [3]:
@torch.inference_mode()
def infer_outputs(frame):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = load_model(cfg, device).eval()
    frame = frame.reset_index(drop=True)
    loader = DataLoader(IWildCamImages(frame, cfg), batch_size=32, num_workers=cfg["inference"]["num_workers"])
    predictions = []
    confidence = []

    for images, _ in loader:
        probabilities = model(normalize(images.to(device))).softmax(1)
        predictions.extend(probabilities.argmax(1).cpu().tolist())
        confidence.extend(probabilities.max(1).values.cpu().tolist())

    names = meta[['y', 'label_name']].drop_duplicates().set_index('y').label_name
    return pd.DataFrame({
        'image_id': frame.image_id,
        'prediction': predictions,
        'prediction_name': pd.Series(predictions).map(names),
        'confidence': confidence,
    })


predictions = infer_outputs(test_meta)
test = test_meta.merge(predictions, on='image_id', validate='one_to_one')
test['model_phantom'] = test.label_name.eq('empty') & ~test.prediction_name.isin(NON_ANIMAL)
hallucinations = test.loc[test.model_phantom]
hallucinations.groupby('split').agg(
    images=('image_id', 'size'),
    cameras=('location_remapped', 'nunique'),
)


,images,cameras
split,,
id_test,60,13
test,951,22


## 4. Camera features

A separate COCO-trained ResNet-50 describes camera appearance. This lets us match cameras without using the classifier's internal features.


In [4]:
class CameraImages(Dataset):
    def __init__(self, metadata, indices):
        self.filenames = metadata.filename.to_list()
        self.indices = indices
        self.transform = Compose([
            Resize((224, 224)),
            ToTensor(),
            Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]), # ensure distribution similar to ImageNet pretraining
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        with Image.open(data_dir / 'train' / self.filenames[index]) as image:
            tensor = self.transform(image.convert('RGB'))
        return tensor, self.indices[index]


In [5]:
@torch.inference_mode()
def extract(metadata, keys, indices):
    model = resnet50(weights=None, norm_layer=FrozenBatchNorm2d)
    model.fc = torch.nn.Identity()
    state = torch.load(WEIGHTS, map_location='cpu', weights_only=True)
    backbone = {}
    for key, value in state.items():
        if key.startswith('backbone.body.'):
            backbone[key.removeprefix('backbone.body.')] = value
    model.load_state_dict(backbone)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.eval().to(device)
    loader = DataLoader(CameraImages(metadata, indices), batch_size=128)
    sums = torch.zeros((len(keys), 2048), device=device)
    counts = np.bincount(indices, minlength=len(keys))

    for images, cameras in loader:
        features = model(images.to(device))
        features = torch.nn.functional.normalize(features, dim=1)
        sums.index_add_(0, cameras.to(device), features)

    means = sums.cpu().numpy() / counts[:, None]
    means /= np.maximum(np.linalg.norm(means, axis=1, keepdims=True), 1e-12)
    return means


In [6]:
appearance_meta = meta.loc[meta.split.isin(['train', 'test', 'id_test'])].reset_index(drop=True)
camera_keys = sorted(set(zip(appearance_meta.split, appearance_meta.location_remapped)))
camera_index = {key: i for i, key in enumerate(camera_keys)}
image_camera_indices = np.array([camera_index[key] for key in
                                zip(appearance_meta.split, appearance_meta.location_remapped)])
means = extract(appearance_meta, camera_keys, image_camera_indices)
print(len(appearance_meta), 'images,', len(camera_keys), 'split/camera means')


180754 images, 455 split/camera means


## 5. Select training sequences

For ID-test, use the same training camera. For unseen cameras, take the nearest training camera by cosine similarity and select sequences labeled with the hallucinated species. The match uses full images, so animals and lighting can affect it too.

The additional sample uses up to 20 random sequences per camera. The image lists are in `results/decisions_expansion/manifest.csv` and `results/decisions_control/manifest.csv`.


In [7]:
training_indices = [i for i, key in enumerate(camera_keys) if key[0] == 'train']
links = []
for (split, camera), group in hallucinations.groupby(['split', 'location_remapped']):
    if split == 'id_test':
        train_camera = camera
        similarity = None
        reason = 'same camera'
    else:
        similarities = means[training_indices] @ means[camera_index[(split, camera)]]
        nearest = int(np.argmax(similarities))
        train_camera = camera_keys[training_indices[nearest]][1]
        similarity = float(similarities[nearest])
        reason = 'nearest appearance'

    for species in sorted(group.prediction.unique()):
        links.append({
            'test_split': split,
            'test_camera': int(camera),
            'train_camera': int(train_camera),
            'rank': 1,
            'cosine_similarity': similarity,
            'reason': reason,
            'hallucinated_y': int(species),
        })

links = pd.DataFrame(links)
links.to_csv(OUT / 'training_camera_links.csv', index=False)
pairs = set(zip(links.train_camera, links.hallucinated_y))
training = train_meta.copy()
training['matches_species'] = [
    (camera, species) in pairs
    for camera, species in zip(training.location_remapped, training.y)
]
selected_sequences = set(training.loc[training.matches_species, 'seq_id'])
extra_ids = pd.concat([
    pd.read_csv(OUT / 'decisions_expansion/manifest.csv'),
    pd.read_csv(OUT / 'decisions_control/manifest.csv'),
]).image_id
candidates = training.loc[training.seq_id.isin(selected_sequences) | training.image_id.isin(extra_ids)].copy()
print(len(candidates), 'images in', candidates.seq_id.nunique(), 'sequences')


21985 images in 4038 sequences


## 6. Labels

Read `results/human_review.json`. `phantom_label` marks an animal-labeled frame judged empty. `not_flagged` includes defaults and does not confirm animal presence. To edit labels, run `python src/server.py` from `pair_b` and open [the labeling tool](http://127.0.0.1:8874/viewer/).

The [WCS dataset](https://lila.science/datasets/wcscameratraps) includes sequence-level labels. Empty frames can inherit an animal label from the sequence.


In [8]:
viewer_manifest = candidates.rename(columns={'location_remapped': 'camera', 'label_name': 'label'}).copy()
viewer_manifest['image_path'] = [os.path.relpath(data_dir / 'train' / name, OUT)
                               for name in viewer_manifest.filename]
viewer_manifest[['image_id', 'seq_id', 'camera', 'datetime', 'label', 'image_path']].to_csv(
    OUT / 'viewer_images.csv', index=False)
print('Viewer input:', OUT / 'viewer_images.csv')
print('Viewer output:', OUT / 'human_review.json')


Viewer input: /home/martin/ai-and-visualization-pair-b-review/phase1/pair_b/results/viewer_images.csv
Viewer output: /home/martin/ai-and-visualization-pair-b-review/phase1/pair_b/results/human_review.json


In [9]:
annotations = pd.DataFrame(load_reviews(), columns=['image_id', 'status'])
annotations = annotations.merge(candidates, on='image_id', validate='one_to_one')
display(annotations.status.value_counts().rename_axis('status').to_frame('images'))


,images
status,
not_flagged,17827
phantom_label,4158


In [10]:
annotated = annotations.copy()
suspects = annotated.loc[annotated.status.eq('phantom_label')]
case_rows = []

for sequence_id, group in suspects.groupby('seq_id'):
    case_rows.append({
        'seq_id': sequence_id,
        'camera': int(group.location_remapped.iloc[0]),
        'species': ', '.join(sorted(group.label_name.unique())),
        'suspect_frames': len(group),
    })

columns = ['seq_id', 'camera', 'species', 'suspect_frames']
cases = pd.DataFrame(case_rows, columns=columns)
cases = cases.sort_values('suspect_frames', ascending=False)
training_summary = {
    'saved_decisions': len(annotated),
    'candidate_images': len(candidates),
    'suspect_frames': len(suspects),
    'suspect_sequences': len(cases),
}
display(pd.Series(training_summary).to_frame('value'))
display(cases.head(12))


,value
saved_decisions,21985
candidate_images,21985
suspect_frames,4158
suspect_sequences,1282


,seq_id,camera,species,suspect_frames
433,3010cc6e-7d42-11eb-8fb5-0242ac1c0002,249,urocyon cinereoargenteus,10
126,3006f482-7d42-11eb-8fb5-0242ac1c0002,221,dasyprocta punctata,10
431,3010c8e0-7d42-11eb-8fb5-0242ac1c0002,249,tapirus bairdii,10
478,3014530c-7d42-11eb-8fb5-0242ac1c0002,259,dasyprocta punctata,10
435,3010d786-7d42-11eb-8fb5-0242ac1c0002,249,agouti paca,10
151,30071e4e-7d42-11eb-8fb5-0242ac1c0002,221,dasyprocta punctata,10
387,300c65de-7d42-11eb-8fb5-0242ac1c0002,160,mazama temama,10
185,30078960-7d42-11eb-8fb5-0242ac1c0002,223,ave desconocida,10
113,3006dc72-7d42-11eb-8fb5-0242ac1c0002,221,urocyon cinereoargenteus,10
494,30167ede-7d42-11eb-8fb5-0242ac1c0002,117,urocyon cinereoargenteus,10


### First and last frames

Compare the first and last animal-labeled frame within each camera, sequence and species group. Skip groups with one frame or tied endpoint timestamps. The chart also shows each camera separately, using at least 10 pairs.


In [11]:
from analyze_sequences import analyze_sequence_ends
from IPython.display import HTML

paired = pd.DataFrame(analyze_sequence_ends())

# Show both directions, taking at most one example per camera in each direction.
examples = paired.loc[paired.first_flag.ne(paired.last_flag)].copy()
examples = examples.drop_duplicates(['camera', 'first_flag', 'last_flag'])
examples = examples.groupby(['first_flag', 'last_flag']).head(3)
examples['inspect'] = [
    f'<a href="http://127.0.0.1:8874/viewer/?sequence={seq}">Inspect sequence</a>'
    for seq in examples.seq_id
]
display(HTML(examples[['camera', 'label', 'first_flag', 'last_flag', 'inspect']].to_html(index=False, escape=False)))


2,513 paired sequence/species groups
First frame with hallucinated label: 232 (9.2%)
Last frame with hallucinated label: 635 (25.3%)
Last minus first: 16.0 percentage points
Only last hallucinated: 483, only first hallucinated: 80


camera,label,first_flag,last_flag,inspect
2,meleagris ocellata,False,True,Inspect sequence
5,papio anubis,False,True,Inspect sequence
6,puma concolor,False,True,Inspect sequence
6,puma concolor,True,False,Inspect sequence
12,bos taurus,True,False,Inspect sequence
19,muntiacus muntjak,True,False,Inspect sequence


### What we see

Open [the atlas](http://127.0.0.1:8874/) to compare cameras and sequences.

- No clear shared pattern between model hallucinations and hallucinated training labels in the map.
- Hallucinated training labels also occur in very dense and very open environments.
- Model hallucinations appear across the map.
- C156 is a clear outlier: it accounts for all 440 test-set Brazilian rabbit hallucinations, spread across 67 sequences.
- Last frames have hallucinated training labels more often than first frames (25.3% vs. 9.2%).
- Strong differences between cameras, with some much worse than others.
